# Sympla — сбор детских/семейных событий (São Paulo)

Запусти ячейки по порядку (▶). Ничего клонировать не нужно.

1. **Установка** зависимостей
2. **Ключ** `ANTHROPIC_API_KEY` (необязательно — из 🔑 Secrets слева; без него работает без LLM)
3. **Движок** — вся логика парсинга/нормализации/валидации (+ офлайн self-test)
4. **Прогон** — discovery + пайплайн → `facts.json`
5. **Просмотр / скачать** результат

> Принцип: факты приносит только детерминированный код; LLM лишь классифицирует поверх текста и не выдумывает.

## 1. Установка зависимостей

In [ ]:
!pip install -q requests beautifulsoup4 lxml anthropic

## 2. Ключ ANTHROPIC_API_KEY (необязательно)
Добавь секрет `ANTHROPIC_API_KEY` в 🔑 (слева) — тогда включится LLM-классификация.
Без ключа всё равно работает (поля age/format и т.п. останутся как из источника).

In [ ]:
import os
def _load_api_key():
    if os.environ.get('ANTHROPIC_API_KEY'):
        return os.environ['ANTHROPIC_API_KEY']
    try:
        from google.colab import userdata
        k = userdata.get('ANTHROPIC_API_KEY')
        if k:
            os.environ['ANTHROPIC_API_KEY'] = k
            return k
    except Exception:
        pass
    return None

USE_LLM = bool(_load_api_key())
print('LLM-классификация:', 'ВКЛ (Haiku)' if USE_LLM else 'ВЫКЛ — нет ANTHROPIC_API_KEY')

## 3. Движок (логика агента + офлайн self-test)
Запусти один раз. В конце прогоняется `selftest()` — должно быть `SELFTEST: OK`.

In [ ]:




import json, re, time, html as _html
from dataclasses import dataclass, field, asdict
from datetime import datetime, date, timedelta
from typing import Any, Optional

import requests
from bs4 import BeautifulSoup

# ----------------------------------------------------------------------------- 
# CONFIG
# ----------------------------------------------------------------------------- 
USER_AGENT = "FamilyEventsBot/0.1 (+contact@example.com)"  # представляться честно
REQUEST_DELAY_SEC = 1.5          # вежливый rate-limit между запросами
REQUEST_TIMEOUT = 20
SOON_WINDOW_HOURS = 48           # «скоро начнётся», если старт в пределах N часов
LLM_MODEL = "claude-haiku-4-5"   # дёшево для классификации; уточнить актуальный id в docs.claude.com
DEFAULT_LANG = "pt-BR"

# TODO[live]: подтвердить реальный механизм листинга, открыв категорию в браузере
# с Network-табом — почти наверняка страница дёргает внутренний JSON-эндпоинт.
# Дёргать его стабильнее, чем парсить HTML-карточки.
CONFIG_LISTING = {
    "base": "https://www.sympla.com.br",
    "city": "São Paulo",
    "city_slug": "sao-paulo-sp",  # сегмент города в URL афиши Sympla
    "categories": ["infantil", "teatros-e-espetaculos", "cursos-e-workshops"],
    "max_pages": 10,              # страховка от бесконечной пагинации
    # "endpoint": "https://www.sympla.com.br/api/.../search?...",  # <- заполнить с живого сайта
}

# Паттерн страницы события Sympla. Историчные формы:
#   /evento/<slug>/<id>   и   /<slug>__<id>
# Ловим обе и отсекаем служебные/листинговые ссылки.
_EVENT_HREF_RE = re.compile(r"sympla\.com\.br/(?:evento/[^?#]+|[^/?#]+__\d+)", re.I)

# Маппинг таксономии Sympla -> внутренние категории (расширять по мере встречи новых)
CATEGORY_MAP = {
    "infantil": "Детские события",
    "teatro": "Детские спектакли",
    "espetáculo": "Детские спектакли",
    "show": "Кино, шоу, концерты для детей",
    "cinema": "Кино, шоу, концерты для детей",
    "workshop": "Мастер-классы",
    "curso": "Мастер-классы",
    "passeio": "Прогулки и туры",
}


# ----------------------------------------------------------------------------- 
# Целевая схема
# ----------------------------------------------------------------------------- 
@dataclass
class Fact:
    title: Optional[str] = None
    description: Optional[str] = None
    start_date: Optional[str] = None   # YYYY-MM-DD
    start_time: Optional[str] = None   # HH:MM
    end_date: Optional[str] = None
    end_time: Optional[str] = None
    address: Optional[str] = None
    city: Optional[str] = None
    district: Optional[str] = None
    age_category: Optional[str] = None
    price: Optional[float] = None
    is_free: Optional[bool] = None
    source_url: Optional[str] = None
    category: Optional[str] = None
    format: Optional[str] = None        # offline | online
    language: Optional[str] = None
    suitable_for_children: Optional[bool] = None
    suitable_for_parents: Optional[bool] = None
    updated_at: Optional[str] = None
    status: Optional[str] = None
    _issues: list = field(default_factory=list)  # служебное: причины «требует проверки»


# ----------------------------------------------------------------------------- 
# Стадия 1 — discovery
# ----------------------------------------------------------------------------- 
def _listing_url(category: str, page: int) -> str:
    base = CONFIG_LISTING["base"].rstrip("/")
    city = CONFIG_LISTING["city_slug"]
    url = f"{base}/eventos/{city}/{category}"
    return f"{url}?page={page}" if page > 1 else url


def _extract_event_links(html_text: str) -> list[str]:
    """Вытащить абсолютные URL страниц событий из HTML листинга (детерминированно).

    Чистый парсинг — отделён от сети, чтобы покрывался оффлайн-тестом фикстурой.
    """
    soup = BeautifulSoup(html_text, "html.parser")
    out: list[str] = []
    seen: set[str] = set()
    for a in soup.find_all("a", href=True):
        href = a["href"].strip()
        if href.startswith("/"):
            href = CONFIG_LISTING["base"].rstrip("/") + href
        if not _EVENT_HREF_RE.search(href):
            continue
        clean = href.split("#")[0].split("?")[0]   # убрать utm/якоря, чтобы дедуп работал
        if clean not in seen:
            seen.add(clean)
            out.append(clean)
    return out


def discover_event_urls(limit: int = 100) -> list[str]:
    """Собрать до `limit` URL событий из категорий Sympla по городу (HTML-листинг + пагинация).

    Стратегия B (HTML-листинг) реализована и детерминирована: ходим по
    /eventos/<город>/<категория>?page=N, тащим ссылки событий, пагинируем пока
    страница приносит что-то новое и не превышен max_pages/limit.

    TODO[live]: egress к sympla.com.br в этой среде закрыт политикой прокси (403),
    поэтому точную форму URL листинга и наличие внутреннего JSON-эндпоинта
    (стратегия A, надёжнее) нельзя подтвердить вживую. Когда доступ появится:
      1) открыть категорию Infantil, во вкладке Network найти search-эндпоинт;
      2) при расхождении поправить _listing_url()/_EVENT_HREF_RE по факту.
    Парсер ссылок (_extract_event_links) от формы листинга не зависит и покрыт тестом.
    """
    found: list[str] = []
    seen: set[str] = set()
    max_pages = int(CONFIG_LISTING.get("max_pages", 10))
    for category in CONFIG_LISTING["categories"]:
        for page in range(1, max_pages + 1):
            if len(found) >= limit:
                return found[:limit]
            try:
                html_text = fetch_html(_listing_url(category, page))
            except requests.RequestException:
                break  # категория недоступна — переходим к следующей
            links = _extract_event_links(html_text)
            fresh = [u for u in links if u not in seen]
            if not fresh:
                break  # пустая/повторная страница — конец пагинации этой категории
            for u in fresh:
                seen.add(u)
                found.append(u)
    return found[:limit]


# ----------------------------------------------------------------------------- 
# Стадия 2 — fetch + parse (JSON-LD first)
# ----------------------------------------------------------------------------- 
_session = requests.Session()
_session.headers.update({"User-Agent": USER_AGENT, "Accept-Language": "pt-BR,pt"})


def fetch_html(url: str) -> str:
    time.sleep(REQUEST_DELAY_SEC)
    r = _session.get(url, timeout=REQUEST_TIMEOUT)
    r.raise_for_status()
    return r.text


def extract_jsonld_events(html_text: str) -> list[dict]:
    """Вернуть все JSON-LD объекты типа Event со страницы (учёт @graph и массивов)."""
    soup = BeautifulSoup(html_text, "html.parser")
    found: list[dict] = []
    for tag in soup.find_all("script", attrs={"type": "application/ld+json"}):
        raw = tag.string or tag.get_text() or ""
        raw = raw.strip()
        if not raw:
            continue
        try:
            data = json.loads(raw)
        except json.JSONDecodeError:
            # некоторые сайты кладут невалидный JSON с комментариями/хвостами — пропускаем
            continue
        for obj in _iter_jsonld_objects(data):
            t = obj.get("@type", "")
            types = t if isinstance(t, list) else [t]
            if any(str(x).endswith("Event") for x in types):
                found.append(obj)
    return found


def _iter_jsonld_objects(data: Any):
    if isinstance(data, list):
        for x in data:
            yield from _iter_jsonld_objects(x)
    elif isinstance(data, dict):
        if "@graph" in data and isinstance(data["@graph"], list):
            for x in data["@graph"]:
                yield from _iter_jsonld_objects(x)
        else:
            yield data


def parse_event(jsonld: dict, source_url: str) -> Fact:
    """Сырое извлечение фактов из JSON-LD (без выдумок: чего нет — то None)."""
    f = Fact(source_url=source_url)
    f.title = _clean(jsonld.get("name"))
    f.description = _clean(jsonld.get("description"))

    sd, st = _split_dt(jsonld.get("startDate"))
    ed, et = _split_dt(jsonld.get("endDate"))
    f.start_date, f.start_time = sd, st
    f.end_date, f.end_time = ed, et

    loc = jsonld.get("location") or {}
    if isinstance(loc, list):
        loc = loc[0] if loc else {}
    addr = loc.get("address") if isinstance(loc, dict) else None
    if isinstance(addr, dict):
        parts = [addr.get("streetAddress"), addr.get("addressLocality")]
        f.address = ", ".join(p for p in parts if p) or loc.get("name")
        f.city = addr.get("addressLocality")
        f.district = addr.get("addressRegion") or None
    elif isinstance(addr, str):
        f.address = addr
    elif isinstance(loc, dict):
        f.address = loc.get("name")

    # offers -> price/currency
    offers = jsonld.get("offers") or {}
    if isinstance(offers, list):
        prices = [_to_float(o.get("price")) for o in offers if isinstance(o, dict)]
        prices = [p for p in prices if p is not None]
        f.price = min(prices) if prices else None     # «от X»
    elif isinstance(offers, dict):
        f.price = _to_float(offers.get("price"))

    # формат из eventAttendanceMode, если есть
    mode = str(jsonld.get("eventAttendanceMode", "")).lower()
    if "online" in mode:
        f.format = "online"
    elif "offline" in mode or "inperson" in mode:
        f.format = "offline"

    # язык напрямую, если размечен
    if jsonld.get("inLanguage"):
        f.language = str(jsonld["inLanguage"])

    # eventStatus -> сразу пометим отмену для стадии normalize
    if str(jsonld.get("eventStatus", "")).endswith("EventCancelled"):
        f.status = "отменено"
    return f


# -----------------------------------------------------------------------------
# Стадия 2b — HTML-фоллбэк (только то, что лежит в стандартных мета-тегах)
# -----------------------------------------------------------------------------
# Принцип фоллбэка: JSON-LD неполный? Дозабираем ТОЛЬКО надёжные, машинно-
# размеченные сигналы (OpenGraph/Twitter/meta + canonical) и явное «бесплатно».
# Даты/адрес из произвольного текста НЕ выдумываем — пусть остаются None, тогда
# стадия 5 честно пометит запись «требует проверки». Это уважает принцип
# «факты приносит только детерминированный код, без догадок».
#
# TODO[live]: egress к sympla.com.br в этой среде закрыт политикой прокси (403),
# поэтому Sympla-специфичные CSS-селекторы карточки нельзя подтвердить вживую.
# Когда доступ появится — добавить сюда точечные селекторы даты/адреса/цены.

_FREE_RE = re.compile(r"\b(gratuito|grátis|gratis|entrada\s+franca|free)\b", re.I)


def parse_html_fallback(html_text: str, source_url: str, base: Optional[Fact] = None) -> Fact:
    """Дозаполнить факт из мета-тегов страницы. Возвращает (возможно тот же) Fact.

    Заполняет ТОЛЬКО пустые (None) поля base — JSON-LD всегда в приоритете.
    """
    f = base or Fact(source_url=source_url)
    soup = BeautifulSoup(html_text, "html.parser")

    def meta(*names_props) -> Optional[str]:
        for key, val in names_props:
            tag = soup.find("meta", attrs={key: val})
            if tag and tag.get("content"):
                return _clean(tag["content"])
        return None

    og_title = meta(("property", "og:title"), ("name", "twitter:title"))
    if not f.title:
        title_tag = soup.find("title")
        f.title = og_title or _clean(title_tag.get_text() if title_tag else None)

    if not f.description:
        f.description = meta(
            ("property", "og:description"),
            ("name", "twitter:description"),
            ("name", "description"),
        )

    # явная бесплатность — надёжный сигнал на бразильских страницах
    if f.is_free is None and f.price is None:
        text = soup.get_text(" ", strip=True)
        if _FREE_RE.search(text):
            f.price, f.is_free = 0.0, True

    # canonical как запасной source_url (исходный URL всё равно приоритетен)
    if not f.source_url:
        canon = soup.find("link", attrs={"rel": "canonical"})
        f.source_url = (canon.get("href") if canon else None) or meta(("property", "og:url"))

    return f


# -----------------------------------------------------------------------------
# Стадия 3 — enrich (LLM)  — классификация ПОВЕРХ добытого текста, без выдумок
# ----------------------------------------------------------------------------- 
ENRICH_SYSTEM = (
    "Ты классифицируешь событие для детского/семейного каталога. "
    "Используй ТОЛЬКО предоставленный текст (title, description, category). "
    "НЕ придумывай факты. Если данных недостаточно — ставь null. "
    "Верни СТРОГО JSON без пояснений и без markdown."
)
ENRICH_INSTRUCTION = """Поля для классификации:
- age_category: строка вроде "4+", "all", "10+" или null
- suitable_for_children: true/false/null
- suitable_for_parents: true/false/null
- language: BCP-47 ("pt-BR") или null
- format: "offline"|"online"|null
- category_internal: одна из [Детские спектакли, Кино, шоу, концерты для детей, Мастер-классы, Прогулки и туры, Детские события] или null
Верни только JSON-объект с этими ключами."""


def enrich_with_llm(f: Fact) -> dict:
    """Классификация недостающих полей. Ленивая загрузка SDK, чтобы оффлайн-тест не требовал ключа."""
    import anthropic  # noqa: локальный импорт намеренно
    client = anthropic.Anthropic()
    payload = {"title": f.title, "description": f.description, "category": f.category}
    resp = client.messages.create(
        model=LLM_MODEL,
        max_tokens=300,
        system=ENRICH_SYSTEM,
        messages=[{"role": "user",
                   "content": ENRICH_INSTRUCTION + "\n\nДанные:\n" + json.dumps(payload, ensure_ascii=False)}],
    )
    text = "".join(b.text for b in resp.content if getattr(b, "type", "") == "text")
    return _safe_json(text)


# ----------------------------------------------------------------------------- 
# Стадия 4 — normalize
# ----------------------------------------------------------------------------- 
def normalize(f: Fact, enrich: dict, today: Optional[date] = None) -> Fact:
    today = today or date.today()

    # доразметка из LLM — только туда, где источник промолчал
    f.age_category = f.age_category or enrich.get("age_category")
    f.suitable_for_children = _first_not_none(f.suitable_for_children, enrich.get("suitable_for_children"))
    f.suitable_for_parents = _first_not_none(f.suitable_for_parents, enrich.get("suitable_for_parents"))
    f.language = f.language or enrich.get("language") or DEFAULT_LANG
    f.format = f.format or enrich.get("format") or "offline"
    f.category = f.category or enrich.get("category_internal") or _map_category(f)

    # is_free из цены
    if f.price is not None:
        f.is_free = (f.price == 0)

    # status (если не выставлен ранее как «отменено»)
    if f.status != "отменено":
        f.status = _derive_status(f, today)

    f.updated_at = datetime.now().replace(microsecond=0).isoformat()
    return f


def _derive_status(f: Fact, today: date) -> str:
    sd = _parse_date(f.start_date)
    ed = _parse_date(f.end_date) or sd
    if sd is None:
        return "требует проверки"
    if ed and ed < today:
        return "прошло"
    start_dt = _parse_dt(f.start_date, f.start_time)
    if start_dt and 0 <= (start_dt - datetime.now()).total_seconds() <= SOON_WINDOW_HOURS * 3600:
        return "скоро начнётся"
    return "новое"   # только что найдено; в очередной прогон станет «актуальное»


# ----------------------------------------------------------------------------- 
# Стадия 5 — validate
# ----------------------------------------------------------------------------- 
REQUIRED = ["title", "start_date", "source_url"]


def validate(f: Fact, check_url: bool = True) -> Fact:
    for key in REQUIRED:
        if not getattr(f, key):
            f._issues.append(f"missing:{key}")

    sd = _parse_date(f.start_date)
    if f.start_date and sd is None:
        f._issues.append("bad_start_date")
    # подозрительно далёкая дата (> 2 лет) — повод перепроверить
    if sd and sd > date.today() + timedelta(days=730):
        f._issues.append("date_too_far")

    if check_url and f.source_url:
        if not _url_ok(f.source_url):
            f._issues.append("url_unreachable")  # ловит выдумки/мёртвые ссылки

    if f._issues:
        f.status = "требует проверки"
    return f


def _url_ok(url: str) -> bool:
    try:
        time.sleep(REQUEST_DELAY_SEC)
        r = _session.get(url, timeout=REQUEST_TIMEOUT, allow_redirects=True, stream=True)
        return r.status_code == 200
    except requests.RequestException:
        return False


# ----------------------------------------------------------------------------- 
# Orchestration
# ----------------------------------------------------------------------------- 
def process_one(url: str, use_llm: bool = True, check_url: bool = True) -> Fact:
    html_text = fetch_html(url)
    return process_html(html_text, url, use_llm=use_llm, check_url=check_url)


def process_html(html_text: str, url: str, use_llm: bool = True, check_url: bool = True) -> Fact:
    """Стадии 2..5 над уже скачанным HTML (вынесено ради оффлайн-тестов)."""
    events = extract_jsonld_events(html_text)
    f = parse_event(events[0], url) if events else Fact(source_url=url)
    # HTML-фоллбэк всегда дозабирает пустые поля; при отсутствии JSON-LD — единственный источник
    f = parse_html_fallback(html_text, url, base=f)
    if not events and not f.title:
        # совсем глухая страница: ни JSON-LD, ни мета — нечего классифицировать
        f.status = "требует проверки"
        f._issues.append("no_jsonld")
        return f
    f.category = _map_category(f)
    enrich = enrich_with_llm(f) if use_llm else {}
    f = normalize(f, enrich)
    f = validate(f, check_url=check_url)
    return f


def run(urls: Optional[list[str]] = None, limit: int = 100, out_path: str = "facts.json",
        use_llm: bool = True, check_url: bool = True) -> dict:
    urls = urls or discover_event_urls(limit)
    urls = urls[:limit]
    facts: list[dict] = []
    for i, url in enumerate(urls, 1):
        try:
            f = process_one(url, use_llm=use_llm, check_url=check_url)
        except Exception as e:           # один битый URL не должен ронять прогон
            f = Fact(source_url=url, status="требует проверки")
            f._issues.append(f"error:{type(e).__name__}")
        rec = {k: v for k, v in asdict(f).items() if not k.startswith("_")}
        facts.append(rec)
        print(f"[{i}/{len(urls)}] {f.status:16} {f.title or url}")
    clean = sum(1 for x in facts if x["status"] != "требует проверки")
    with open(out_path, "w", encoding="utf-8") as fp:
        json.dump(facts, fp, ensure_ascii=False, indent=2)
    summary = {"total": len(facts), "clean": clean, "needs_review": len(facts) - clean, "out": out_path}
    print("SUMMARY:", summary)
    return summary


# ----------------------------------------------------------------------------- 
# helpers
# ----------------------------------------------------------------------------- 
def _clean(s):
    if not s:
        return None
    return _html.unescape(re.sub(r"\s+", " ", str(s))).strip() or None

def _split_dt(value):
    """'2026-06-14T15:00:00-03:00' -> ('2026-06-14','15:00'); '2026-06-14' -> (date, None)."""
    if not value:
        return None, None
    s = str(value)
    m = re.match(r"(\d{4}-\d{2}-\d{2})(?:[T ](\d{2}:\d{2}))?", s)
    if not m:
        return None, None
    return m.group(1), m.group(2)

def _to_float(v):
    if v in (None, "", "0.00") and v != 0:
        # "0.00" — валидный ноль; разрулим ниже
        pass
    try:
        return float(str(v).replace(",", "."))
    except (TypeError, ValueError):
        return None

def _parse_date(s):
    try:
        return datetime.strptime(s, "%Y-%m-%d").date() if s else None
    except (TypeError, ValueError):
        return None

def _parse_dt(d, t):
    if not d:
        return None
    try:
        return datetime.strptime(f"{d} {t or '00:00'}", "%Y-%m-%d %H:%M")
    except ValueError:
        return None

def _map_category(f: Fact) -> Optional[str]:
    hay = " ".join(x for x in [f.category, f.title, f.description] if x).lower()
    for key, val in CATEGORY_MAP.items():
        if key in hay:
            return val
    return None

def _first_not_none(*vals):
    for v in vals:
        if v is not None:
            return v
    return None

def _safe_json(text: str) -> dict:
    text = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.MULTILINE).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return {}


# -----------------------------------------------------------------------------
# Оффлайн self-test — логика парсинга/нормализации/валидации без сети и без LLM.
# CLAUDE.md: «сохранять зелёным». Запуск:  python3 sympla_agent.py --selftest
# -----------------------------------------------------------------------------
_FIXTURE_JSONLD = """<html><head>
<script type="application/ld+json">
{"@context":"https://schema.org","@type":"Event",
 "name":"Teatro Infantil: O Pequeno Príncipe",
 "description":"Espetáculo para a família, a partir de 4 anos.",
 "startDate":"2026-07-15T15:00:00-03:00","endDate":"2026-07-15T16:30:00-03:00",
 "eventAttendanceMode":"https://schema.org/OfflineEventAttendanceMode",
 "eventStatus":"https://schema.org/EventScheduled",
 "location":{"@type":"Place","name":"Teatro Paulo Autran",
   "address":{"@type":"PostalAddress","streetAddress":"Praca Roosevelt, 210","addressLocality":"Sao Paulo","addressRegion":"SP"}},
 "offers":{"@type":"Offer","price":"40.00","priceCurrency":"BRL"},
 "inLanguage":"pt-BR"}
</script></head><body></body></html>"""

# JSON-LD без описания/цены — проверяем, что HTML-фоллбэк дозабирает мета + «Gratuito»
_FIXTURE_PARTIAL = """<html><head>
<meta property="og:title" content="Oficina de Desenho para Crianças">
<meta property="og:description" content="Workshop gratuito de desenho, 6+">
<script type="application/ld+json">
{"@context":"https://schema.org","@type":"Event",
 "name":"Oficina de Desenho para Crianças",
 "startDate":"2026-08-01","location":{"@type":"Place","name":"SESC"}}
</script></head><body><p>Entrada: Gratuito. Vagas limitadas.</p></body></html>"""

_FIXTURE_LISTING = """<html><body>
<a href="/evento/o-pequeno-principe/1234567">card</a>
<a href="https://www.sympla.com.br/teatro-infantil__2222?utm_source=x">card</a>
<a href="/evento/o-pequeno-principe/1234567#hero">dup</a>
<a href="/eventos/sao-paulo-sp/infantil">not-an-event</a>
<a href="https://outrosite.com/evento/abc/9">other-domain</a>
</body></html>"""


def selftest() -> int:
    fails: list[str] = []

    def check(cond, msg):
        if not cond:
            fails.append(msg)

    today = date(2026, 6, 29)

    # 1) JSON-LD путь
    evs = extract_jsonld_events(_FIXTURE_JSONLD)
    check(len(evs) == 1, "jsonld: ожидался 1 Event")
    f = process_html(_FIXTURE_JSONLD, "https://www.sympla.com.br/evento/x/1",
                     use_llm=False, check_url=False)
    check(f.title == "Teatro Infantil: O Pequeno Príncipe", f"title={f.title!r}")
    check(f.start_date == "2026-07-15" and f.start_time == "15:00", f"dt={f.start_date} {f.start_time}")
    check(f.price == 40.0 and f.is_free is False, f"price={f.price} free={f.is_free}")
    check(f.format == "offline", f"format={f.format}")
    check(f.language == "pt-BR", f"lang={f.language}")
    check(f.category == "Детские события", f"cat={f.category}")
    check(f.status != "требует проверки" and not f._issues, f"status={f.status} issues={f._issues}")

    # 2) HTML-фоллбэк дозабирает пустые поля (описание из og, бесплатность из текста)
    f2 = process_html(_FIXTURE_PARTIAL, "https://www.sympla.com.br/oficina__2222",
                      use_llm=False, check_url=False)
    check(f2.description == "Workshop gratuito de desenho, 6+", f"desc={f2.description!r}")
    check(f2.is_free is True and f2.price == 0.0, f"free={f2.is_free} price={f2.price}")
    check(f2.category == "Мастер-классы", f"cat2={f2.category}")

    # 3) глухая страница без JSON-LD и без мета -> требует проверки
    f3 = process_html("<html><body>nada</body></html>", "https://x/y",
                      use_llm=False, check_url=False)
    check(f3.status == "требует проверки" and "no_jsonld" in f3._issues, f"empty={f3.status} {f3._issues}")

    # 4) discovery-парсер ссылок: дедуп, абсолютизация, отсев чужого домена/листинга
    links = _extract_event_links(_FIXTURE_LISTING)
    check(links == [
        "https://www.sympla.com.br/evento/o-pequeno-principe/1234567",
        "https://www.sympla.com.br/teatro-infantil__2222",
    ], f"links={links}")

    # 5) статусы: прошло / скоро / отменено / нет даты
    past = normalize(Fact(title="t", source_url="u", start_date="2020-01-01"), {}, today=today)
    check(past.status == "прошло", f"past={past.status}")
    nodate = validate(normalize(Fact(title="t", source_url="u"), {}, today=today), check_url=False)
    check(nodate.status == "требует проверки", f"nodate={nodate.status}")
    cancelled = Fact(title="t", source_url="u", start_date="2026-07-15", status="отменено")
    cancelled = normalize(cancelled, {}, today=today)
    check(cancelled.status == "отменено", f"cancelled={cancelled.status}")

    # 6) валидация ловит битую дату и слишком далёкую
    bad = validate(Fact(title="t", source_url="u", start_date="2026-13-40"), check_url=False)
    check("bad_start_date" in bad._issues, f"baddate={bad._issues}")

    if fails:
        print("SELFTEST: FAIL")
        for m in fails:
            print("  -", m)
        return 1
    print("SELFTEST: OK (6 групп проверок пройдено)")
    return 0

selftest()  # офлайн-проверка логики без сети и LLM


## 4. Прогон: discovery + пайплайн → facts.json
Правь `LIMIT` / город / категории при необходимости.

In [ ]:
import os
LIMIT = 20
# LLM не используется, если нет ключа ANTHROPIC_API_KEY -> USE_LLM=False
USE_LLM = bool(os.environ.get('ANTHROPIC_API_KEY'))

# при желании поменяй город/категории:
# CONFIG_LISTING['city_slug'] = 'rio-de-janeiro-rj'
# CONFIG_LISTING['categories'] = ['infantil']

print('Собираю ссылки из листинга Sympla...')
try:
    urls = discover_event_urls(limit=LIMIT)
except Exception as e:
    urls = []
    print('Ошибка discovery:', type(e).__name__, e)
print('Найдено ссылок:', len(urls))

if urls:
    summary = run(urls=urls, limit=LIMIT, use_llm=USE_LLM, check_url=True)
else:
    print(
        '\nПусто — листинг, скорее всего, рендерится через JavaScript/внутренний JSON.\n'
        'Варианты:\n'
        "  1) Открой https://www.sympla.com.br/eventos/" + CONFIG_LISTING['city_slug'] + "/infantil ,\n"
        '     DevTools -> Network, найди search-эндпоинт (JSON со списком событий).\n'
        '  2) Или задай ссылки вручную и запусти напрямую:\n'
        "       run(urls=['https://www.sympla.com.br/evento/.../123'], use_llm=USE_LLM)\n"
        '  3) Или подними рендеринг: !pip install playwright && playwright install chromium\n'
    )

## 5. Просмотр и скачивание результата

In [ ]:
import json, pandas as pd
try:
    data = json.load(open('facts.json', encoding='utf-8'))
    df = pd.DataFrame(data)
    print('Всего:', len(df))
    if 'status' in df:
        print(df['status'].value_counts())
    display(df.head(20))
    from google.colab import files
    files.download('facts.json')
except FileNotFoundError:
    print('facts.json ещё нет — сначала выполни ячейку 4 с непустым списком ссылок.')